# 02 — Prepare Training Pairs

**Run once after `01_embed_and_block.ipynb`. Safe to re-run — output tensors are skipped if they already exist on Drive.**

What this notebook does:
1. Loads train embeddings from Drive
2. Builds FAISS blocking indices for the train target pool (S2 + S3)
3. Stratified-samples 100k S1 entities by country
4. Runs dual-modality FAISS blocking for the sampled S1 entities
5. Builds (positive, hard-negative) pair tensors with scalar features
6. Saves `all_name_inputs.npy`, `all_addr_inputs.npy`, `all_scalars.npy`,
   `all_labels.npy`, `all_groups.npy` to `DATASET_ROOT/output/embeddings/`

Downstream notebooks that depend on these outputs:
- `03_train.ipynb` — loads pair tensors, trains one fold per runtime

In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────────
!pip install -q transformers sentencepiece faiss-cpu

import os
try:
    import google.colab
    COLAB = True
except ImportError:
    COLAB = False

BASE_PATH = '/content/drive/MyDrive/Amazon/student_resource 2'  # EDIT

if COLAB and BASE_PATH.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = os.path.expanduser(BASE_PATH)
assert os.path.isdir(DATA_DIR), f'Folder not found at {DATA_DIR}'
print('Dataset OK at', DATA_DIR)

WORK_ROOT = '/content/er'
os.makedirs(WORK_ROOT, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Colab ready.')

## Section 0 — Configuration

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import gc
import faiss
from tqdm import tqdm

In [ ]:
try:
    COLAB
except NameError:
    COLAB = False; DATA_DIR = None; WORK_ROOT = None

if COLAB:
    DATASET_ROOT = Path(DATA_DIR)
    ROOT = Path(WORK_ROOT)
else:
    ROOT = Path.cwd()
    if not (ROOT / 'dataset').exists():
        ROOT = Path('..').resolve()
    DATASET_ROOT = ROOT

TRAIN = DATASET_ROOT / 'dataset/train'
EMB   = DATASET_ROOT / 'output/embeddings'
EMB.mkdir(parents=True, exist_ok=True)

# ── Config ───────────────────────────────────────────────────────────────────
SEED             = 42
TOP_K_FAISS      = 80
NEGATIVE_RATIO   = 5
TRAIN_S1_SAMPLE  = 100_000
QUANTIZED_INDEX  = True

is_cuda_available = torch.cuda.is_available()
DEVICE = torch.device('cuda' if is_cuda_available else 'cpu')
print(f'Device: {DEVICE}')
np.random.seed(SEED); torch.manual_seed(SEED)

## Section 2 — FAISS Utilities

In [ ]:
def build_faiss_index(embeddings_fp16, use_gpu=False, quantize=QUANTIZED_INDEX):
    """
    Build a FAISS index over L2-normalized embeddings.
    Default: SQ8 scalar quantization → ~4x less RAM than IndexFlatIP.
    Inner product on L2-normalized vectors == cosine similarity.
    """
    emb = embeddings_fp16.astype(np.float32)
    D = emb.shape[1]
    if quantize and not use_gpu:
        index = faiss.IndexScalarQuantizer(D, faiss.ScalarQuantizer.QT_8bit)
    else:
        index = faiss.IndexFlatIP(D)
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    if isinstance(index, faiss.IndexScalarQuantizer):
        index.train(emb)
    index.add(emb)
    return index


def dual_faiss_search(s1_names, s1_addrs, name_index, addr_index, target_ids, k=80):
    """Dual-modality FAISS search. Returns dict: {local_s1_idx: set(target_entity_ids)}"""
    name_scores, name_indices = name_index.search(s1_names.astype(np.float32), k)
    addr_scores, addr_indices = addr_index.search(s1_addrs.astype(np.float32), k)

    result = {}
    for i in range(len(s1_names)):
        candidates = set()
        for idx in name_indices[i]:
            if 0 <= idx < len(target_ids):
                candidates.add(target_ids[idx])
        for idx in addr_indices[i]:
            if 0 <= idx < len(target_ids):
                candidates.add(target_ids[idx])
        result[i] = candidates
    return result


def country_filter(candidates_dict, s1_countries, target_country_map, allow_mismatch=False):
    """Filter candidates by country match."""
    if allow_mismatch:
        return candidates_dict
    filtered = {}
    for local_idx, cand_set in candidates_dict.items():
        s1_country = str(s1_countries[local_idx]).lower().strip() if local_idx < len(s1_countries) else ''
        filtered_cands = set()
        for cid in cand_set:
            cand_country = str(target_country_map.get(cid, '')).lower().strip()
            if s1_country == cand_country or s1_country == '' or cand_country == '':
                filtered_cands.add(cid)
        filtered[local_idx] = filtered_cands
    return filtered


def load_country_map(paths):
    """Returns dict: entity_id → country"""
    result = {}
    for p in paths:
        for chunk in pd.read_csv(p, sep='\t', chunksize=100_000,
                                  usecols=['entity_id', 'country'], low_memory=False):
            for _, row in chunk.iterrows():
                result[str(row['entity_id'])] = str(row['country'])
    return result

## Check: skip if pair tensors already exist

In [ ]:
PAIR_FILES = [
    EMB / 'all_name_inputs.npy',
    EMB / 'all_addr_inputs.npy',
    EMB / 'all_scalars.npy',
    EMB / 'all_labels.npy',
    EMB / 'all_groups.npy',
]

if all(f.exists() for f in PAIR_FILES):
    print('All pair tensor files already exist on Drive — nothing to do.')
    print('Delete them from Drive if you want to rebuild from scratch.')
    raise SystemExit('Skipping rebuild.')  # harmless: just stops cell execution
else:
    print('Pair tensor files not found. Building...')

## Section 2 — Load Train Embeddings and Build FAISS Indices

In [ ]:
print('Loading train target embeddings...')
train_s2_name = np.load(EMB / 'train_s2_name.npy')
train_s3_name = np.load(EMB / 'train_s3_name.npy')
train_s2_addr = np.load(EMB / 'train_s2_addr.npy')
train_s3_addr = np.load(EMB / 'train_s3_addr.npy')
train_s2_ids  = np.load(EMB / 'train_s2_name_ids.npy', allow_pickle=True).tolist()
train_s3_ids  = np.load(EMB / 'train_s3_name_ids.npy', allow_pickle=True).tolist()

# Combine S2 + S3 into one target pool
train_target_name = np.concatenate([train_s2_name, train_s3_name], axis=0)
train_target_addr = np.concatenate([train_s2_addr, train_s3_addr], axis=0)
train_target_ids  = train_s2_ids + train_s3_ids

print(f'Train target pool: {len(train_target_ids):,} records')

train_target_country_map = load_country_map([
    TRAIN / 'train_source2.tsv', TRAIN / 'train_source3.tsv'
])

print('Building train FAISS indices...')
train_name_index = build_faiss_index(train_target_name)
train_addr_index = build_faiss_index(train_target_addr)
print('FAISS indices ready.')

## Section 3A — Stratified S1 Sample

In [ ]:
def stratified_sample_s1(tsv_path, n=TRAIN_S1_SAMPLE, seed=SEED):
    """
    Two-pass stratified sample of S1 entity_ids by country.
    Returns list of sampled entity_ids.
    """
    rng = np.random.default_rng(seed)
    country_ids = defaultdict(list)
    for chunk in pd.read_csv(tsv_path, sep='\t', chunksize=100_000,
                              usecols=['entity_id', 'country'], low_memory=False):
        for _, row in chunk.iterrows():
            country_ids[str(row['country'])].append(str(row['entity_id']))
    total = sum(len(v) for v in country_ids.values())
    sampled = []
    for country, ids in country_ids.items():
        quota = max(1, round(n * len(ids) / total))
        chosen = rng.choice(ids, size=min(quota, len(ids)), replace=False)
        sampled.extend(chosen.tolist())
    print(f'Sampled {len(sampled):,} S1 entities (target {n:,})')
    print({c: len(ids) for c, ids in country_ids.items()})
    return sampled

sampled_s1_ids = stratified_sample_s1(TRAIN / 'train_source1.tsv')

## Section 3B — Load Ground Truth and S1 Embeddings

In [ ]:
# Load ground truth
print('Loading ground truth...')
gt_df = pd.read_csv(TRAIN / 'train_ground_truth.tsv', sep='\t')
gt_df = gt_df[gt_df['matched_entity_ids'].notna() & (gt_df['matched_entity_ids'] != '')]
expl = gt_df.assign(_id=gt_df['matched_entity_ids'].str.split(',')).explode('_id')
expl['_id'] = expl['_id'].str.strip()
expl = expl[expl['_id'] != '']
gt_map = defaultdict(set)
for _sid, _val in zip(expl['source1_entity_id'], expl['_id']):
    gt_map[_sid].add(_val)

total_s1 = 0
for chunk in pd.read_csv(TRAIN / 'train_source1.tsv', sep='\t', chunksize=100_000,
                         usecols=['entity_id']):
    total_s1 += len(chunk)

print(f'Ground truth: {len(gt_map):,} S1 entities with ≥1 match '
      f'({len(gt_map)/max(total_s1,1):.2%} of {total_s1:,} train S1)')

In [ ]:
# Load S1 embeddings
train_s1_name = np.load(EMB / 'train_s1_name.npy')
train_s1_addr = np.load(EMB / 'train_s1_addr.npy')
train_s1_ids  = np.load(EMB / 'train_s1_name_ids.npy', allow_pickle=True).tolist()
train_s1_country_map = load_country_map([TRAIN / 'train_source1.tsv'])

train_s1_id_to_idx     = {eid: i for i, eid in enumerate(train_s1_ids)}
train_target_id_to_idx = {eid: i for i, eid in enumerate(train_target_ids)}

sampled_s1_idx = [train_s1_id_to_idx[sid] for sid in sampled_s1_ids if sid in train_s1_id_to_idx]

In [ ]:
# Run FAISS blocking for sampled S1
print(f'Running FAISS blocking for {len(sampled_s1_idx):,} sampled S1 entities...')
sampled_s1_name     = train_s1_name[sampled_s1_idx]
sampled_s1_addr     = train_s1_addr[sampled_s1_idx]
sampled_s1_countries = [train_s1_country_map.get(train_s1_ids[i], '') for i in sampled_s1_idx]

candidates_dict = dual_faiss_search(
    sampled_s1_name, sampled_s1_addr,
    train_name_index, train_addr_index,
    train_target_ids, k=TOP_K_FAISS
)
candidates_dict = country_filter(candidates_dict, sampled_s1_countries, train_target_country_map)
print('FAISS blocking complete.')

## Section 3C — Feature Vector Construction

In [ ]:
def pair_features(s1_name_emb, s1_addr_emb, cand_name_emb, cand_addr_emb,
                   s1_country, cand_country):
    """
    Build input feature vector for the Tower+Adapter model.

    name_input = concat(s1_name_emb, cand_name_emb)  → 1536-dim
    addr_input = concat(s1_addr_emb, cand_addr_emb)  → 1536-dim
    scalars    = [cos_name, cos_addr, country_eq]     → 3-dim
    """
    name_input = np.concatenate([s1_name_emb, cand_name_emb]).astype(np.float32)
    addr_input = np.concatenate([s1_addr_emb, cand_addr_emb]).astype(np.float32)
    cos_name   = float(np.dot(s1_name_emb, cand_name_emb))
    cos_addr   = float(np.dot(s1_addr_emb, cand_addr_emb))
    country_eq = float(str(s1_country).lower().strip() == str(cand_country).lower().strip())
    scalars = np.array([cos_name, cos_addr, country_eq], dtype=np.float32)
    return name_input, addr_input, scalars

## Section 3D — Build Training Pairs

In [ ]:
print('Building training pairs...')
all_name_inputs = []
all_addr_inputs = []
all_scalars     = []
all_labels      = []
all_groups      = []  # S1 entity_id per pair (for macro-F0.5)

rng = np.random.default_rng(SEED)

for local_idx, global_idx in enumerate(tqdm(sampled_s1_idx, desc='Building pairs')):
    s1_id        = train_s1_ids[global_idx]
    s1_name_emb  = train_s1_name[global_idx].astype(np.float32)
    s1_addr_emb  = train_s1_addr[global_idx].astype(np.float32)
    s1_country   = train_s1_country_map.get(s1_id, '')

    candidates   = candidates_dict.get(local_idx, set())
    true_matches = gt_map.get(s1_id, set())

    positives = candidates & true_matches
    negatives = list(candidates - true_matches)

    n_neg = min(len(negatives), len(positives) * NEGATIVE_RATIO)
    sampled_neg = rng.choice(negatives, size=n_neg, replace=False) if n_neg > 0 else []

    for cand_id in positives:
        if cand_id not in train_target_id_to_idx:
            continue
        cand_idx      = train_target_id_to_idx[cand_id]
        cand_name_emb = train_target_name[cand_idx].astype(np.float32)
        cand_addr_emb = train_target_addr[cand_idx].astype(np.float32)
        cand_country  = train_target_country_map.get(cand_id, '')
        ni, ai, sc = pair_features(s1_name_emb, s1_addr_emb,
                                    cand_name_emb, cand_addr_emb,
                                    s1_country, cand_country)
        all_name_inputs.append(ni); all_addr_inputs.append(ai)
        all_scalars.append(sc);     all_labels.append(1.0)
        all_groups.append(s1_id)

    for cand_id in sampled_neg:
        if cand_id not in train_target_id_to_idx:
            continue
        cand_idx      = train_target_id_to_idx[cand_id]
        cand_name_emb = train_target_name[cand_idx].astype(np.float32)
        cand_addr_emb = train_target_addr[cand_idx].astype(np.float32)
        cand_country  = train_target_country_map.get(cand_id, '')
        ni, ai, sc = pair_features(s1_name_emb, s1_addr_emb,
                                    cand_name_emb, cand_addr_emb,
                                    s1_country, cand_country)
        all_name_inputs.append(ni); all_addr_inputs.append(ai)
        all_scalars.append(sc);     all_labels.append(0.0)
        all_groups.append(s1_id)

all_name_inputs = np.array(all_name_inputs, dtype=np.float32)
all_addr_inputs = np.array(all_addr_inputs, dtype=np.float32)
all_scalars     = np.array(all_scalars, dtype=np.float32)
all_labels      = np.array(all_labels, dtype=np.float32)
all_groups      = np.array(all_groups)

print(f'Training pairs: {len(all_labels):,}  '
      f'(pos: {all_labels.sum():.0f}, neg: {(1-all_labels).sum():.0f})')

## Save Pair Tensors to Drive

In [ ]:
np.save(EMB / 'all_name_inputs.npy', all_name_inputs)
np.save(EMB / 'all_addr_inputs.npy', all_addr_inputs)
np.save(EMB / 'all_scalars.npy',     all_scalars)
np.save(EMB / 'all_labels.npy',      all_labels)
np.save(EMB / 'all_groups.npy',      all_groups)

print('Pair tensors saved to Drive:')
for f in PAIR_FILES:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:30s}  {size_mb:.1f} MB')

## Done

Pair tensors written to `DATASET_ROOT/output/embeddings/`:

```
all_name_inputs.npy   shape: (N, 1536)  float32
all_addr_inputs.npy   shape: (N, 1536)  float32
all_scalars.npy       shape: (N, 3)     float32
all_labels.npy        shape: (N,)       float32
all_groups.npy        shape: (N,)       object (S1 entity_id strings)
```

Next step: run `03_train.ipynb` — launch 5 copies simultaneously, one per fold.